# NB_patient_360

Builds `adb_caresync.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance providers and per-patient lab result aggregates. Full rebuild on every run.

In [0]:
%sql
-- Full rebuild: one row per patient (driving table silver.patients, LEFT JOINs only)
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary. Source: STTM silver->gold.'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                    AS patient_id,
  p.first_name                                                                 AS first_name,
  p.last_name                                                                  AS last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,   -- silver column is TIMESTAMP
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                                     AS gender,
  p.city                                                                       AS city,
  p.state                                                                      AS state,
  CAST(p.registered_hospital_id AS INT)                                        AS hospital_id,
  h.hospital_name                                                              AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                         AS insurance_provider_id,
  ip.provider_name                                                             AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                     AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                             AS critical_test_count,
  la.last_test_date                                                            AS last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:
%sql
-- Row count must equal silver.patients (no rows dropped/duplicated); test totals must reconcile with silver.lab_results
SELECT
  (SELECT COUNT(*) FROM adb_caresync.silver.patients)            AS silver_patients,
  COUNT(*)                                                       AS gold_rows,
  COUNT(DISTINCT patient_id)                                     AS distinct_patients,
  SUM(total_tests)                                               AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results
    WHERE patient_id IN (SELECT patient_id FROM adb_caresync.silver.patients)) AS silver_tests_for_known_patients,
  COUNT_IF(hospital_name IS NULL)                                AS missing_hospital_name,
  COUNT_IF(insurance_provider_name IS NULL)                      AS missing_insurance_name,
  COUNT_IF(total_tests = 0)                                      AS patients_without_tests
FROM adb_caresync.gold.patient_360;

In [0]:
%sql
SELECT * FROM adb_caresync.gold.patient_360 ORDER BY patient_id LIMIT 20;

In [0]:
# Return the gold table's record count to the caller (e.g. a job task or dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))